# Day 20: Fine‑Tuning BERT for Text Classification

Welcome back! After understanding attention and the Transformer architecture on Day 19, today you'll put that knowledge into practice by fine‑tuning a pre‑trained Transformer (BERT/DistilBERT) for a real‑world text classification task.

---

**Goal**: Learn an efficient, industry‑practical workflow to fine‑tune a pre‑trained Transformer for sentence‑level classification using Hugging Face `transformers`.

**Topics Covered**
- Why fine‑tune pre‑trained Transformers (transfer learning in NLP)
- Tokenization, [CLS] representation, and classification heads
- Building a minimal, reliable training pipeline with `Trainer`
- Evaluating with accuracy/F1 and running quick inference
- Practical tips: small datasets, freezing layers, batch sizes on CPU/GPU

**Real‑World Impact**: This workflow powers production systems like sentiment analysis, spam detection, intent recognition, and content moderation.

**Prerequisites**: Day 15–19 (Neural Nets → PyTorch → RNNs/LSTMs → Transformers & Attention).



## 1. Concept Overview

### Why fine‑tune a pre‑trained Transformer?
Pre‑trained models (BERT, RoBERTa, DistilBERT) learn powerful language representations from massive corpora. Fine‑tuning adapts these representations to your task with relatively little labeled data. This is transfer learning for NLP.

- **When to use**: You have a text classification task (sentiment, spam, intent) and limited labeled data; you need strong performance fast.
- **Why it works**: The model already understands syntax/semantics. You only train a thin classification head (and optionally the encoder) to specialize for your labels.

### How the architecture maps to classification
- **Tokenizer** converts text to token IDs with attention masks.
- **Encoder** (e.g., DistilBERT) produces contextual embeddings for each token.
- **[CLS] token** (or pooled output) summarizes the sequence for classification.
- **Classification head** (a small feed‑forward layer) predicts class logits.

### Practical decisions that matter
- **Model choice**: Start with `distilbert-base-uncased` for speed; move to `bert-base-uncased` if you need extra accuracy.
- **Max sequence length**: 128 or 256 is usually enough for sentence/short paragraph tasks.
- **Batch size**: Fit what your hardware allows; gradient accumulation can simulate larger batches.
- **Freezing layers**: On very small datasets, freezing early layers can reduce overfitting.
- **Metrics**: For balanced binary tasks, accuracy is fine; for imbalanced data, track F1.
- **Data hygiene**: Clean labels, remove duplicates/leakage, standardize casing if using cased/uncased models.

### Common pitfalls
- Training for too many epochs → overfitting (watch validation metrics).
- Forgetting to use attention masks → degraded performance.
- Mismatched label mapping between train/eval/predict.
- Not setting a seed → hard to reproduce results.



## 2. Code Demo: Fine‑Tuning DistilBERT on a Classification Task
We will fine‑tune `distilbert-base-uncased` on a small subset of GLUE SST‑2 (binary sentiment). The setup uses Hugging Face `datasets` and `transformers` with `Trainer` for clarity and minimal code.


In [1]:
# 2.1 Environment Setup and Imports
import os
import random
import numpy as np

import torch
from packaging import version

# Optional: silence warnings for a clean demo
import warnings
warnings.filterwarnings("ignore")

# Check core deps and device
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")  # Apple Silicon
    return torch.device("cpu")

device = get_device()
print(f"PyTorch: {torch.__version__} | Device: {device}")

# Hugging Face libs
try:
    from datasets import load_dataset
    from transformers import (
        AutoTokenizer,
        AutoModelForSequenceClassification,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
    )
    import evaluate
    HF_AVAILABLE = True
except Exception as e:
    HF_AVAILABLE = False
    print("Hugging Face libraries not available.")
    print("Install with: pip install transformers datasets evaluate")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device.type == "cuda":
    torch.cuda.manual_seed_all(SEED)


PyTorch: 2.9.0+cpu | Device: cpu


### 2.2 Load a Small SST‑2 Subset
We use GLUE SST‑2 (binary sentiment). To keep runtime light, we sample a small subset for training and evaluation. Adjust sizes if you have a GPU.


In [2]:
if not HF_AVAILABLE:
    raise RuntimeError("Please install transformers, datasets, evaluate to run this section.")

raw_ds = load_dataset("glue", "sst2")

# Keep it light for demo: small random subsamples
train_sample_size = 2000
val_sample_size = 1000

# Shuffle deterministically and select a subset
raw_train = raw_ds["train"].shuffle(seed=SEED).select(range(min(train_sample_size, len(raw_ds["train"]))))
raw_val = raw_ds["validation"].shuffle(seed=SEED).select(range(min(val_sample_size, len(raw_ds["validation"]))))

label_names = ["negative", "positive"]
num_labels = 2
print(raw_train[0])
print(f"Train size: {len(raw_train)} | Val size: {len(raw_val)}")


{'sentence': 'klein , charming in comedies like american pie and dead-on in election , ', 'label': 1, 'idx': 32326}
Train size: 2000 | Val size: 872


### 2.3 Tokenization and Data Collation
We use `AutoTokenizer` with truncation and dynamic padding via `DataCollatorWithPadding` for efficient batches.


In [4]:
model_name = "distilbert-base-uncased"
max_length = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(example):
    return tokenizer(
        example["sentence"],
        truncation=True,
        max_length=max_length,
    )

train_ds = raw_train.map(preprocess, batched=True, remove_columns=["sentence", "idx"])
val_ds = raw_val.map(preprocess, batched=True, remove_columns=["sentence", "idx"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds.features)


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

{'label': ClassLabel(names=['negative', 'positive']), 'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8'))}


### 2.4 Model, TrainingArguments, and Trainer
We attach a classification head to DistilBERT and train for a couple of epochs with evaluation each epoch.


In [7]:
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Metrics: accuracy + F1 (if available)
accuracy = evaluate.load("accuracy")
try:
    f1 = evaluate.load("f1")
except Exception:
    f1 = None

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    results = {"accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"]}
    if f1 is not None and num_labels == 2:
        results["f1"] = f1.compute(predictions=preds, references=labels, average="binary")["f1"]
    return results

output_dir = "./day20-distilbert-sst2"
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
eval_metrics = trainer.evaluate()

print("Eval:", eval_metrics)


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
50,0.603900
100,0.356900
150,0.299600
200,0.246100
250,0.218700


Eval: {'eval_loss': 0.34543323516845703, 'eval_accuracy': 0.8681192660550459, 'eval_f1': 0.8709315375982043, 'eval_runtime': 56.6342, 'eval_samples_per_second': 15.397, 'eval_steps_per_second': 0.494, 'epoch': 2.0}


### 2.5 Quick Inference
Use the fine‑tuned model to classify new sentences.


In [9]:
def predict_sentences(texts):
    enc = tokenizer(texts, truncation=True, padding=True, max_length=max_length, return_tensors="pt").to(device)
    model.to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = probs.argmax(axis=-1)
    return preds, probs

samples = [
    "I absolutely loved this!",
    "This was disappointing and boring.",
    "Not bad, not great—just okay.",
]
labels = [id2label[i] for i in range(num_labels)]

preds, probs = predict_sentences(samples)
for s, p, pr in zip(samples, preds, probs):
    print(f"{s}\n  → Pred: {id2label[int(p)]} | Prob: {pr}")


I absolutely loved this!
  → Pred: positive | Prob: [0.02985621 0.97014374]
This was disappointing and boring.
  → Pred: negative | Prob: [0.9531688  0.04683121]
Not bad, not great—just okay.
  → Pred: negative | Prob: [0.5543739  0.44562608]


## 3. Hands‑on Exercise
Tweak the pipeline to see how choices affect performance and speed. Keep runs short; focus on iteration and observation.

- Exercise 1: Swap the base model and compare metrics
- Exercise 2: Freeze encoder layers to fight overfitting
- Exercise 3: Tune learning rate and epochs
- Exercise 4: Try a different dataset
- Exercise 5: Add a lightweight error analysis

> Visual suggestion: small table template for logging LR, epochs, accuracy, F1, notes.


### Exercise 1 — Swap the base model
Try `bert-base-uncased` or `roberta-base` (needs a different tokenizer/model pair), keep all else the same, and compare metrics.


In [ ]:
# TODO: Swap model and rerun training/eval
# Hint: For RoBERTa use 'roberta-base' and ensure tokenizer/model names match
# model_name = "bert-base-uncased"
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForSequenceClassification.from_pretrained(
#     model_name, num_labels=num_labels, id2label=id2label, label2id=label2id
# )
# Re‑map datasets if tokenizer changed:
# train_ds = raw_train.map(preprocess, batched=True, remove_columns=["sentence", "idx"])
# val_ds = raw_val.map(preprocess, batched=True, remove_columns=["sentence", "idx"])
# trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
#                   tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
# trainer.train(); print(trainer.evaluate())


### Exercise 2 Freeze encoder layers
Freeze backbone layers (e.g., all of DistilBERT) to train only the classification head. This reduces overfitting on tiny datasets and speeds training.


In [ ]:
# TODO: Freeze encoder parameters
# For DistilBERT backbones, names usually start with 'distilbert.'
# For BERT backbones, names start with 'bert.'
# for name, param in model.named_parameters():
#     if name.startswith('distilbert.'):
#         param.requires_grad = False
# print(sum(p.requires_grad for p in model.parameters()), "trainable params groups")
# trainer.train(); print(trainer.evaluate())


### Exercise 3: Tune learning rate and epochs
Try `learning_rate` in {1e‑5, 2e‑5, 5e‑5} and `num_train_epochs` in {2, 3, 4}. Keep notes on metrics and overfitting signs.


In [ ]:
# TODO: Modify TrainingArguments and re‑train quickly
# training_args = TrainingArguments(
#     output_dir=output_dir,
#     per_device_train_batch_size=16,
#     per_device_eval_batch_size=32,
#     num_train_epochs=3,        # try 2/3/4
#     learning_rate=5e-5,        # try 1e-5 / 2e-5 / 5e-5
#     weight_decay=0.01,
#     evaluation_strategy="epoch",
#     logging_strategy="steps",
#     logging_steps=50,
#     save_strategy="no",
#     seed=SEED,
# )
# trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
#                   tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
# trainer.train(); print(trainer.evaluate())


### Exercise 4: Try a different dataset
Swap SST‑2 with `ag_news` (4‑class). Replace the text field and adjust `num_labels` and `label_names` accordingly. Keep the rest identical to isolate the effect of data.


In [11]:
# TODO: Minimal AG News setup (uncomment to try)
# raw_ds = load_dataset("ag_news")
# label_names = ["World", "Sports", "Business", "Sci/Tech"]
# num_labels = 4
# def preprocess_ag(example):
#     return tokenizer(example["text"], truncation=True, max_length=max_length)
# raw_train = raw_ds["train"].shuffle(seed=SEED).select(range(4000))
# raw_val = raw_ds["test"].shuffle(seed=SEED).select(range(2000))
# train_ds = raw_train.map(preprocess_ag, batched=True, remove_columns=["text"])
# val_ds = raw_val.map(preprocess_ag, batched=True, remove_columns=["text"])
# model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
# trainer = Trainer(model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
#                   tokenizer=tokenizer, data_collator=data_collator, compute_metrics=compute_metrics)
# trainer.train(); print(trainer.evaluate())


### Exercise 5: Lightweight error analysis
Collect a few misclassified examples and inspect them. Are they ambiguous? Noisy? Out‑of‑distribution? This informs data cleaning and model limits.


In [12]:
# TODO: Inspect misclassifications on validation set
# preds = trainer.predict(val_ds)
# logits = preds.predictions
# y_true = preds.label_ids
# y_pred = logits.argmax(axis=-1)
# mistakes = []
# for i, (yt, yp) in enumerate(zip(y_true, y_pred)):
#     if yt != yp:
#         mistakes.append(i)
# # Show a few mistakes
# for idx in mistakes[:5]:
#     print("TEXT:", raw_val[idx]["sentence"])  # adjust field for other datasets
#     print("TRUE:", id2label[int(y_true[idx])], "PRED:", id2label[int(y_pred[idx])])
#     print("---")


## 4. Takeaways + Next Steps

### Key Takeaways
- Pre‑trained Transformers give you strong baselines with minimal labeled data.
- Fine‑tuning = tokenize → encode → [CLS]/pooled → classification head → train briefly.
- Start with DistilBERT for speed; scale to BERT/RoBERTa if needed.
- Watch validation metrics to avoid overfitting; adjust LR/epochs and consider freezing.
- Maintain reproducibility (seeds) and verify label mappings end‑to‑end.

### Next Steps
- Try sequence tagging (token classification) for NER with `AutoModelForTokenClassification`.
- Explore encoder‑decoder models (e.g., T5) for summarization or translation.
- Add experiment tracking (Weights & Biases) and error analysis dashboards.
- Learn dataset curation best practices and prompt‑based evaluation for modern pipelines.


---

Well done. You’ve taken the Transformer concepts from Day 19 and turned them into a production‑oriented fine‑tuning workflow. On Day 21, we’ll expand into token‑level tasks (like NER) or sequence‑to‑sequence tasks (like summarization), building on this foundation.


---

## 📫 Let's Connect.
- 💼 **LinkedIn:** [hashirahmed07](https://www.linkedin.com/in/hashirahmed07/)
- 📧 **Email:** [Hashirahmad330@gmail.com](mailto:Hashirahmad330@gmail.com)
- 🐙 **GitHub:** [CodeByHashir](https://github.com/CodeByHashir)
